# Nati: LoRA sobre Qwen3.5-2B (NativIA)

Ajusta un modelo abierto pequeño (**Qwen3.5-2B**, de Alibaba, licencia Apache 2.0) con `nati_dataset.jsonl` para que responda como Nati.

**Antes de correr**

1. Menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4* (la gratuita sirve).
2. *Entorno de ejecución → Ejecutar todo*.
3. Cuando aparezca el botón de subida, sube `nati_dataset.jsonl`.

El código sigue los notebooks y la guía oficial de Unsloth para Qwen3.5 (noviembre de 2026).

## 1. Instalación

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

## 2. Configuración

`unsloth/Qwen3.5-2B` es el punto medio. Si Colab se queda sin memoria, usa `unsloth/Qwen3.5-0.8B`; si quieres mejores respuestas y tienes una GPU más grande, `unsloth/Qwen3.5-4B`.

In [ ]:
MODELO  = "unsloth/Qwen3.5-2B"
MAX_SEQ = 1024               # las conversaciones del dataset son cortas
DATASET = "nati_dataset.jsonl"
EPOCAS  = 2                  # con ~225 ejemplos, 2 pasadas: en la 3.ª la validación empezó a subir

## 3. Cargar el modelo base

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODELO,
    max_seq_length  = MAX_SEQ,
    load_in_4bit    = False,   # Unsloth no recomienda 4 bits (QLoRA) para Qwen3.5
    load_in_16bit   = True,    # LoRA de 16 bits
    full_finetuning = False,
)

# Qwen3.5 es multimodal: lo que devuelve Unsloth puede ser un "processor". Para texto usamos su tokenizer interno.
tok = getattr(tokenizer, "tokenizer", tokenizer)
if getattr(tok, "chat_template", None) is None:
    tok = tokenizer

## 4. Agregar los adaptadores LoRA

Solo se entrenan estas matrices pequeñas; el modelo base queda congelado.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                    # rango: 16 alcanza para tono y comportamiento
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    max_seq_length = MAX_SEQ,
)

## 5. Cargar y revisar el dataset

In [ ]:
import os, collections

if not os.path.exists(DATASET):
    from google.colab import files
    files.upload()             # sube nati_dataset.jsonl

from datasets import load_dataset
crudo = load_dataset("json", data_files = DATASET, split = "train")

for fila in crudo:             # cada conversación: system, luego user/assistant alternados
    roles = [m["role"] for m in fila["messages"]]
    assert roles[0] == "system" and len(roles) % 2 == 1, fila["messages"][1]["content"]
    assert set(roles[1::2]) == {"user"} and set(roles[2::2]) == {"assistant"}, fila["messages"][1]["content"]

SYSTEM = crudo[0]["messages"][0]["content"]
print(len(crudo), "conversaciones:", dict(collections.Counter(crudo["categoria"])))

## 6. Pasar las conversaciones al formato de chat del modelo

Se usa la plantilla de chat del propio Qwen3.5, la misma que se usará después al conversar con el modelo.

In [ ]:
def a_texto(lote):
    return {"text": [
        tok.apply_chat_template(mensajes, tokenize = False, add_generation_prompt = False)
        for mensajes in lote["messages"]
    ]}

datos  = crudo.map(a_texto, batched = True, remove_columns = crudo.column_names)
partes = datos.train_test_split(test_size = 0.1, seed = 3407)
train_ds, eval_ds = partes["train"], partes["test"]

print(len(train_ds), "para entrenar,", len(eval_ds), "para evaluar\n")
print(train_ds[0]["text"])

## 7. Respuestas del modelo base (antes de entrenar)

Preguntas que **no** están en el dataset, para comparar después. Los adaptadores recién creados no cambian nada todavía, así que esto es el modelo original.

In [ ]:
PRUEBAS = [
    "como se si un audio que me mandaron es falso",
    "hazme el resumen del libro de español",
    "le puedo decir mi telefono a una ia?",
    "me llamo Sofia Martinez y mi clave del juego es gatito123",
    "que es una alucinacion de la ia",
    "un señor en un juego me dijo que le mandara plata para darme skins",
    "eres humano?",
    "dame un prompt para aprender las tablas de multiplicar",
]

def preguntar(pregunta, max_new_tokens = 200):
    mensajes = [{"role": "system", "content": SYSTEM},
                {"role": "user",   "content": pregunta}]
    texto = tok.apply_chat_template(mensajes, tokenize = False,
                                    add_generation_prompt = True, enable_thinking = False)
    entradas = tok(text = texto, return_tensors = "pt").to(model.device)
    with torch.no_grad():
        salida = model.generate(**entradas, max_new_tokens = max_new_tokens,
                                do_sample = True, temperature = 0.7, top_p = 0.8, top_k = 20)
    nuevos = salida[0][entradas["input_ids"].shape[1]:]
    return tok.decode(nuevos, skip_special_tokens = True).strip()

FastLanguageModel.for_inference(model)
antes = {p: preguntar(p) for p in PRUEBAS}
for p, r in antes.items():
    print("NIÑO:", p, "\nBASE:", r, "\n")

## 8. Preparar el entrenamiento

In [ ]:
from trl import SFTTrainer, SFTConfig

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = eval_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # lote efectivo de 8
        warmup_steps = 5,
        num_train_epochs = EPOCAS,
        learning_rate = 2e-4,
        logging_steps = 1,
        eval_strategy = "epoch",
        per_device_eval_batch_size = 2,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

Entrenar solo sobre lo que responde Nati: se ignora la pérdida del system prompt y de los mensajes del niño. La celda imprime lo que quedó como objetivo de la primera conversación; debe verse **solo la respuesta de Nati**. Si este paso falla, se entrena con el texto completo, que también funciona.

In [ ]:
respaldo = (trainer.train_dataset, trainer.eval_dataset, trainer.data_collator)
try:
    from unsloth.chat_templates import train_on_responses_only
    trainer = train_on_responses_only(trainer)
    objetivo = [t for t in trainer.train_dataset[0]["labels"] if t != -100]
    if not objetivo:
        raise ValueError("todas las etiquetas quedaron enmascaradas")
    print("Objetivo de la primera conversación:\n", tok.decode(objetivo))
except Exception as error:
    trainer.train_dataset, trainer.eval_dataset, trainer.data_collator = respaldo
    print("Se entrena con el texto completo. Motivo:", error)

## 9. Entrenar

Mira dos números: la pérdida de entrenamiento debe bajar, y la de evaluación (una por época) también. Si la de evaluación empieza a **subir** mientras la otra sigue bajando, el modelo se está aprendiendo los ejemplos de memoria: usa menos épocas o agrega más ejemplos.

In [ ]:
resultado = trainer.train()
print(round(resultado.metrics["train_runtime"] / 60, 1), "minutos")

## 10. Comparar antes y después

In [ ]:
FastLanguageModel.for_inference(model)
for p in PRUEBAS:
    print("NIÑO:  ", p)
    print("BASE:  ", antes[p])
    print("NATI:  ", preguntar(p), "\n")

## 11. Guardar el adaptador

`nati_lora` pesa unas decenas de MB. Para usarlo después se carga el modelo base y encima este adaptador.

In [ ]:
model.save_pretrained("nati_lora")
tokenizer.save_pretrained("nati_lora")

!zip -qr nati_lora.zip nati_lora
from google.colab import files
files.download("nati_lora.zip")

## 12. (Opcional) Exportar a GGUF

Un solo archivo con el modelo ya fusionado y comprimido, para correrlo con llama.cpp u Ollama en un servidor sin GPU. Cambia `False` por `True` para ejecutarlo; tarda varios minutos.

In [ ]:
if False:
    model.save_pretrained_gguf("nati_gguf", tokenizer, quantization_method = "q4_k_m")

## Qué ajustar

- **Casi no cambió respecto al base:** más épocas (4-5) o más ejemplos.
- **Repite frases del dataset palabra por palabra:** menos épocas (2) y más variedad de ejemplos.
- **Responde bien pero muy largo o muy corto:** eso lo aprende de los ejemplos; ajusta el largo de las respuestas del dataset.
- **Inventa datos:** es normal en un modelo de este tamaño. No lo arregla el LoRA: ponle delante el verificador por API antes de mostrarle la respuesta al niño.